1. Import Libraries and Ensure Target Directory Exists

In [15]:
import os
import pandas as pd

2. Carete new directories to store clean dataset

In [16]:
os.makedirs("../data/cleaned", exist_ok=True)

3. Load the Raw Marketing Dataset

In [17]:
marketing_clean = pd.read_csv("../data/raw/marketing_raw.csv")

4. Clean Text Columns & Standardize Categories

In [18]:
text_cols = [
    "campaign_id",
    "campaign_name",
    "campaign_type",
    "channel",
    "target_segment",
]

5. Strip whitespace from string columns

In [19]:
for col in text_cols:
  marketing_clean[col] = marketing_clean[col].astype(str).str.strip()

6. Handle missing campaign_type converted from 'nan' string back or imputed

In [20]:
marketing_clean["campaign_type"] = (
    marketing_clean["campaign_type"].replace("nan", "Other").fillna("Other")
)

7. Impute Missing Clicks Using Impressions & CTR Formula
8. Formula: clicks = round(impressions * ctr)

In [21]:
calculated_clicks = (
    marketing_clean["impressions"] * marketing_clean["ctr"]
).round()
marketing_clean["clicks"] = (
    marketing_clean["clicks"].fillna(calculated_clicks).astype(int)
)

9. Recalculate ROAS for Formula Consistency
10. Formula: roas = revenue_generated / spend

In [22]:
marketing_clean["roas"] = (
    marketing_clean["revenue_generated"] / marketing_clean["spend"]
).round(2)

11. Datetime Parsing

In [23]:
marketing_clean["campaign_date"] = pd.to_datetime(
    marketing_clean["campaign_date"], errors="coerce"
)


12. Verify Cleaning Results

In [24]:
print("Dataset Shape:", marketing_clean.shape)
print("Remaining Missing Values:\n", marketing_clean.isnull().sum())
print("Campaign Types:\n", marketing_clean["campaign_type"].value_counts())

Dataset Shape: (50000, 14)
Remaining Missing Values:
 campaign_id          0
campaign_date        0
campaign_name        0
campaign_type        0
channel              0
target_segment       0
impressions          0
clicks               0
conversions          0
spend                0
revenue_generated    0
ctr                  0
conversion_rate      0
roas                 0
dtype: int64
Campaign Types:
 campaign_type
Display       8390
Search        8372
Email         8357
Influencer    8254
Social        8220
Affiliate     8207
Other          200
Name: count, dtype: int64


13.  Export Cleaned Dataset to 'cleaned/' Directory

In [25]:
output_path = os.path.join("../data/cleaned", "marketing_cleaned.csv")
marketing_clean.to_csv(output_path, index=False)